### 📝 Task info: 
- Build a machine learning classification model to predict whether a bank customer will subscribe to a term deposit (yes or no) based on the customer's demographic information, financial details, and previous marketing campaign interactions.

### 💻 Code Work:

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 📤 0. Data Loading (from db load tables data as pandas df)

In [6]:
!pip install mysql-connector-python
import pandas as pd
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="ABCD",
    database="bank_marketing"
)

# Load validated table data into Pandas DataFrame
df = pd.read_sql("SELECT * FROM bank_marketing", conn)


C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\3938222366.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM bank_marketing", conn)


In [7]:
df = pd.read_sql("SELECT * FROM bank_marketing", conn)

df.head()

C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\1597814708.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM bank_marketing", conn)


,age,job,marital,education,default_status,balance,housing,loan,contact,day,month,duration_minutes,campaign,previous,poutcome,y,pdays
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,4.35,1,0,unknown,no,-1
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5,may,2.52,1,0,unknown,no,-1
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5,may,1.27,1,0,unknown,no,-1
3,47,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5,may,1.53,1,0,unknown,no,-1
4,33,unknown,single,unknown,no,1.0,no,no,unknown,5,may,3.30,1,0,unknown,no,-1


In [8]:
df.shape

(45211, 17)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               45211 non-null  int64  
 1   job               45211 non-null  str    
 2   marital           45211 non-null  str    
 3   education         45211 non-null  str    
 4   default_status    45211 non-null  str    
 5   balance           45211 non-null  float64
 6   housing           45211 non-null  str    
 7   loan              45211 non-null  str    
 8   contact           45211 non-null  str    
 9   day               45211 non-null  int64  
 10  month             45211 non-null  str    
 11  duration_minutes  45211 non-null  float64
 12  campaign          45211 non-null  int64  
 13  previous          45211 non-null  int64  
 14  poutcome          45211 non-null  str    
 15  y                 45211 non-null  str    
 16  pdays             45211 non-null  int64  
dtypes: f

In [10]:
df.isnull().sum()

age                 0
job                 0
marital             0
education           0
default_status      0
balance             0
housing             0
loan                0
contact             0
day                 0
month               0
duration_minutes    0
campaign            0
previous            0
poutcome            0
y                   0
pdays               0
dtype: int64

### 👩🏻‍💻 1. Modeling

### 🛠️ 1.1 Pre-Requisites

#### 🎯 1.1.1 X & y:

In [11]:
# Selecting target variable (y)
y = df['y']

# Selecting input features (X)
X = df.drop('y', axis=1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (45211, 16)
y shape: (45211,)


In [12]:
print(X.columns)
print(y.value_counts())

Index(['age', 'job', 'marital', 'education', 'default_status', 'balance',
       'housing', 'loan', 'contact', 'day', 'month', 'duration_minutes',
       'campaign', 'previous', 'poutcome', 'pdays'],
      dtype='str')
y
no     39922
yes     5289
Name: count, dtype: int64


In [13]:
y = y.map({'yes': 1, 'no': 0})

#### ✂️ 1.1.2 Train-Test Split:

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [15]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (36168, 16)
X_test : (9043, 16)
y_train: (36168,)
y_test : (9043,)


#### 🔄 1.1.3 Missing Values & Outliers Handling 

In [16]:
print("Missing values in X_train:")
print(X_train.isnull().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum())

Missing values in X_train:
age                 0
job                 0
marital             0
education           0
default_status      0
balance             0
housing             0
loan                0
contact             0
day                 0
month               0
duration_minutes    0
campaign            0
previous            0
poutcome            0
pdays               0
dtype: int64

Missing values in X_test:
age                 0
job                 0
marital             0
education           0
default_status      0
balance             0
housing             0
loan                0
contact             0
day                 0
month               0
duration_minutes    0
campaign            0
previous            0
poutcome            0
pdays               0
dtype: int64


In [17]:
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Missing values in y_train: 0
Missing values in y_test: 0


In [18]:
numeric_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

print(numeric_cols)

Index(['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous',
       'pdays'],
      dtype='str')


In [19]:
X_train[numeric_cols].describe()

,age,balance,day,duration_minutes,campaign,previous,pdays
count,36168.000000,36168.00000,36168.000000,36168.000000,36168.000000,36168.000000,36168.000000
mean,40.892999,1365.49342,15.817961,4.308462,2.763935,0.581730,40.157238
std,10.627075,3068.54350,8.331980,4.319049,3.104161,2.408766,100.162614
min,18.000000,-8019.00000,1.000000,0.000000,1.000000,0.000000,-1.000000
25%,33.000000,74.00000,8.000000,1.720000,1.000000,0.000000,-1.000000
50%,39.000000,451.00000,16.000000,3.000000,2.000000,0.000000,-1.000000
75%,48.000000,1430.25000,21.000000,5.300000,3.000000,0.000000,-1.000000
max,95.000000,102127.00000,31.000000,81.970000,63.000000,275.000000,871.000000


In [20]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

print(numeric_cols)

Index(['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous',
       'pdays'],
      dtype='str')


In [21]:
Q1 = X_train[numeric_cols].quantile(0.25)
Q3 = X_train[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outlier_count = (
    (X_train[numeric_cols] < lower_limit) |
    (X_train[numeric_cols] > upper_limit)
).sum()

print("Outlier count:")
print(outlier_count)

Outlier count:
age                  394
balance             3770
day                    0
duration_minutes    2632
campaign            2465
previous            6584
pdays               6584
dtype: int64


In [22]:
outlier_percentage = (outlier_count / len(X_train)) * 100

print("Outlier percentage:")
print(outlier_percentage.round(2))

Outlier percentage:
age                  1.09
balance             10.42
day                  0.00
duration_minutes     7.28
campaign             6.82
previous            18.20
pdays               18.20
dtype: float64


In [23]:
outlier_summary = pd.DataFrame({
    'Q1': Q1,
    'Q3': Q3,
    'IQR': IQR,
    'Lower Limit': lower_limit,
    'Upper Limit': upper_limit,
    'Outlier Count': outlier_count,
    'Outlier %': outlier_percentage
})

outlier_summary

,Q1,Q3,IQR,Lower Limit,Upper Limit,Outlier Count,Outlier %
age,33.00,48.00,15.00,10.500,70.500,394,1.089361
balance,74.00,1430.25,1356.25,-1960.375,3464.625,3770,10.423579
day,8.00,21.00,13.00,-11.500,40.500,0,0.000000
duration_minutes,1.72,5.30,3.58,-3.650,10.670,2632,7.277151
campaign,1.00,3.00,2.00,-2.000,6.000,2465,6.815417
previous,0.00,0.00,0.00,0.000,0.000,6584,18.203937
pdays,-1.00,-1.00,0.00,-1.000,-1.000,6584,18.203937


#### ⚙️ 1.1.4 Feature Engineering of x for y

In [24]:
X_train['previously_contacted'] = (X_train['pdays'] != -1).astype(int)
X_test['previously_contacted'] = (X_test['pdays'] != -1).astype(int)

In [25]:
print(y_train.value_counts())

y
0    31937
1     4231
Name: count, dtype: int64


In [26]:
print(y_train.value_counts(normalize=True) * 100)

y
0    88.301814
1    11.698186
Name: proportion, dtype: float64


In [27]:
numeric_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("Numerical columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns:
['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous', 'pdays', 'previously_contacted']

Categorical columns:
['job', 'marital', 'education', 'default_status', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\3297479729.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [28]:
print(y_train.unique())
print(y_train.value_counts())

[0 1]
y
0    31937
1     4231
Name: count, dtype: int64


In [29]:
y_train_num = y_train.copy()

In [30]:
print(y_train_num.value_counts())

y
0    31937
1     4231
Name: count, dtype: int64


In [31]:
numeric_corr = X_train[numeric_cols].corrwith(y_train_num)

print(numeric_corr.sort_values(ascending=False))

duration_minutes        0.396743
previously_contacted    0.163589
pdays                   0.100703
previous                0.088847
balance                 0.055025
age                     0.024704
day                    -0.026595
campaign               -0.071978
dtype: float64


In [32]:
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print(categorical_cols)

['job', 'marital', 'education', 'default_status', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\1565242246.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [33]:
from sklearn.feature_selection import mutual_info_classif

X_cat = X_train[categorical_cols].copy()

# Convert categorical columns to category codes
for col in categorical_cols:
    X_cat[col] = X_cat[col].astype('category').cat.codes

mi_scores = mutual_info_classif(
    X_cat,
    y_train,
    random_state=42
)

mi_results = pd.Series(
    mi_scores,
    index=categorical_cols
).sort_values(ascending=False)

print(mi_results)


poutcome          0.034261
month             0.026366
contact           0.017271
housing           0.016324
job               0.009224
education         0.007090
marital           0.004511
loan              0.002536
default_status    0.000000
dtype: float64


In [34]:
numeric_corr

age                     0.024704
balance                 0.055025
day                    -0.026595
duration_minutes        0.396743
campaign               -0.071978
previous                0.088847
pdays                   0.100703
previously_contacted    0.163589
dtype: float64

In [35]:
mi_results

poutcome          0.034261
month             0.026366
contact           0.017271
housing           0.016324
job               0.009224
education         0.007090
marital           0.004511
loan              0.002536
default_status    0.000000
dtype: float64

In [36]:
print("Numerical Feature Correlation:")
print(numeric_corr.sort_values(ascending=False))

print("\nCategorical Feature Mutual Information:")
print(mi_results.sort_values(ascending=False))

Numerical Feature Correlation:
duration_minutes        0.396743
previously_contacted    0.163589
pdays                   0.100703
previous                0.088847
balance                 0.055025
age                     0.024704
day                    -0.026595
campaign               -0.071978
dtype: float64

Categorical Feature Mutual Information:
poutcome          0.034261
month             0.026366
contact           0.017271
housing           0.016324
job               0.009224
education         0.007090
marital           0.004511
loan              0.002536
default_status    0.000000
dtype: float64


In [37]:
selected_features = [
    'age',
    'balance',
    'day',
    'duration_minutes',
    'campaign',
    'previous',
    'pdays',
    'job',
    'marital',
    'education',
    'default_status',
    'housing',
    'loan',
    'contact',
    'month',
    'poutcome',
    'previously_contacted'
]

In [38]:
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [39]:
print(X_train.columns.tolist())

['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous', 'pdays', 'job', 'marital', 'education', 'default_status', 'housing', 'loan', 'contact', 'month', 'poutcome', 'previously_contacted']


In [40]:
X_train['previously_contacted'] = (X_train['pdays'] != -1).astype(int)

X_test['previously_contacted'] = (X_test['pdays'] != -1).astype(int)

In [41]:
print('previously_contacted' in X_train.columns)
print('previously_contacted' in X_test.columns)

True
True


In [42]:
selected_features = [
    'age',
    'balance',
    'day',
    'duration_minutes',
    'campaign',
    'previous',
    'pdays',
    'job',
    'marital',
    'education',
    'default_status',
    'housing',
    'loan',
    'contact',
    'month',
    'poutcome',
    'previously_contacted'
]

In [43]:
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [44]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print(X_train.columns.tolist())

X_train shape: (36168, 17)
X_test shape: (9043, 17)
['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous', 'pdays', 'job', 'marital', 'education', 'default_status', 'housing', 'loan', 'contact', 'month', 'poutcome', 'previously_contacted']


In [45]:
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['job', 'marital', 'education', 'default_status', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\325403764.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [46]:
numeric_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print("Numerical columns:")
print(numeric_cols)

Numerical columns:
['age', 'balance', 'day', 'duration_minutes', 'campaign', 'previous', 'pdays', 'previously_contacted']


In [47]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    drop='first',
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

In [48]:
encoded_cols = encoder.get_feature_names_out(categorical_cols)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_cols,
    index=X_test.index
)

In [49]:
X_train_final = pd.concat(
    [X_train[numeric_cols], X_train_cat],
    axis=1
)

X_test_final = pd.concat(
    [X_test[numeric_cols], X_test_cat],
    axis=1
)

In [50]:
print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

print(X_train_final.dtypes)

X_train_final shape: (36168, 43)
X_test_final shape: (9043, 43)
age                       int64
balance                 float64
day                       int64
duration_minutes        float64
campaign                  int64
previous                  int64
pdays                     int64
previously_contacted      int64
job_blue-collar         float64
job_entrepreneur        float64
job_housemaid           float64
job_management          float64
job_retired             float64
job_self-employed       float64
job_services            float64
job_student             float64
job_technician          float64
job_unemployed          float64
job_unknown             float64
marital_married         float64
marital_single          float64
education_secondary     float64
education_tertiary      float64
education_unknown       float64
default_status_yes      float64
housing_yes             float64
loan_yes                float64
contact_telephone       float64
contact_unknown         float64
month_au

In [51]:
from sklearn.preprocessing import StandardScaler

In [52]:
scaler = StandardScaler()

In [53]:
X_train_scaled = scaler.fit_transform(X_train_final)

X_test_scaled = scaler.transform(X_test_final)

In [54]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train_final.columns,
    index=X_train_final.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test_final.columns,
    index=X_test_final.index
)

In [55]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (36168, 43)
X_test_scaled shape: (9043, 43)


In [56]:
X_train_scaled.describe().loc[['mean', 'std']]

,age,balance,day,duration_minutes,campaign,previous,pdays,previously_contacted,job_blue-collar,job_entrepreneur,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
mean,-1.194454e-16,-2.043144e-17,8.801237e-17,3.762136e-17,4.557784e-17,-1.493067e-17,2.514639e-17,-2.612867e-17,3.241527e-18,2.986134e-17,...,1.257320e-17,-1.414485e-17,2.514639e-17,3.830896e-17,8.172577e-17,1.748460e-17,4.950696e-17,2.436057e-17,4.714948e-17,-3.889832e-17
std,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,...,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00


#### 🤖 1.2 Model + Define, Train & Study

#### Logistic Regression

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True
        ), categorical_cols)
    ]
)

print("Preprocessor ready!")

Preprocessor ready!


In [60]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=500,
        solver='liblinear',
        random_state=42
    ))
])

logreg_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [61]:
# Predictions on training and test data

y_train_pred_logreg = logreg_model.predict(X_train)
y_test_pred_logreg = logreg_model.predict(X_test)

print("Logistic Regression predictions completed!")

Logistic Regression predictions completed!


In [62]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Classification metrics
train_accuracy = accuracy_score(y_train, y_train_pred_logreg)
test_accuracy = accuracy_score(y_test, y_test_pred_logreg)

train_precision = precision_score(y_train, y_train_pred_logreg)
test_precision = precision_score(y_test, y_test_pred_logreg)

train_recall = recall_score(y_train, y_train_pred_logreg)
test_recall = recall_score(y_test, y_test_pred_logreg)

train_f1 = f1_score(y_train, y_train_pred_logreg)
test_f1 = f1_score(y_test, y_test_pred_logreg)

# ROC-AUC
y_test_prob_logreg = logreg_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, y_test_prob_logreg)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

print("Train Precision:", train_precision)
print("Test Precision:", test_precision)

print("Train Recall:", train_recall)
print("Test Recall:", test_recall)

print("Train F1:", train_f1)
print("Test F1:", test_f1)

print("Test ROC-AUC:", test_roc_auc)

Train Accuracy: 0.9021787215217872
Test Accuracy: 0.9013601680858122
Train Precision: 0.6553115194979829
Test Precision: 0.6456140350877193
Train Recall: 0.34554478846608366
Test Recall: 0.34782608695652173
Train F1: 0.4524914887031879
Test F1: 0.4520884520884521
Test ROC-AUC: 0.9055540101774002


#### 
- Logistic Regression: The model achieved a Test F1-score of 0.4521 and ROC-AUC of 0.9056. The close train and test performance indicates good generalization with no significant overfitting. However, the relatively low recall of 0.3478 shows that the model misses a considerable number of positive customers.

### KNN 

In [63]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

knn_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ))
])

knn_model.fit(X_train, y_train)

print("KNN trained successfully!")

KNN trained successfully!


In [64]:
y_train_pred_knn = knn_model.predict(X_train)
y_test_pred_knn = knn_model.predict(X_test)

print("KNN predictions completed!")

KNN predictions completed!


In [65]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Evaluation
train_accuracy_knn = accuracy_score(y_train, y_train_pred_knn)
test_accuracy_knn = accuracy_score(y_test, y_test_pred_knn)

train_precision_knn = precision_score(y_train, y_train_pred_knn)
test_precision_knn = precision_score(y_test, y_test_pred_knn)

train_recall_knn = recall_score(y_train, y_train_pred_knn)
test_recall_knn = recall_score(y_test, y_test_pred_knn)

train_f1_knn = f1_score(y_train, y_train_pred_knn)
test_f1_knn = f1_score(y_test, y_test_pred_knn)

# ROC-AUC
y_test_prob_knn = knn_model.predict_proba(X_test)[:, 1]
test_roc_auc_knn = roc_auc_score(y_test, y_test_prob_knn)

print("Train Accuracy:", train_accuracy_knn)
print("Test Accuracy:", test_accuracy_knn)

print("Train Precision:", train_precision_knn)
print("Test Precision:", test_precision_knn)

print("Train Recall:", train_recall_knn)
print("Test Recall:", test_recall_knn)

print("Train F1:", train_f1_knn)
print("Test F1:", test_f1_knn)

print("Test ROC-AUC:", test_roc_auc_knn)

Train Accuracy: 0.9223623092236231
Test Accuracy: 0.8966051089240297
Train Precision: 0.7571376942537044
Test Precision: 0.6016528925619835
Train Recall: 0.4951548097376507
Test Recall: 0.3440453686200378
Train F1: 0.5987424978565304
Test F1: 0.4377630787733013
Test ROC-AUC: 0.8276067011279419


### Navie bayes

In [66]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Naive Bayes preprocessing
preprocessor_nb = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Naive Bayes model
nb_model = Pipeline([
    ('preprocessor', preprocessor_nb),
    ('model', MultinomialNB())
])

# Train
nb_model.fit(X_train, y_train)

print("Naive Bayes trained successfully!")

Naive Bayes trained successfully!


In [67]:
y_train_pred_nb = nb_model.predict(X_train)
y_test_pred_nb = nb_model.predict(X_test)

print("Naive Bayes predictions completed!")

Naive Bayes predictions completed!


In [68]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_accuracy_nb = accuracy_score(y_train, y_train_pred_nb)
test_accuracy_nb = accuracy_score(y_test, y_test_pred_nb)

train_precision_nb = precision_score(y_train, y_train_pred_nb)
test_precision_nb = precision_score(y_test, y_test_pred_nb)

train_recall_nb = recall_score(y_train, y_train_pred_nb)
test_recall_nb = recall_score(y_test, y_test_pred_nb)

train_f1_nb = f1_score(y_train, y_train_pred_nb)
test_f1_nb = f1_score(y_test, y_test_pred_nb)

y_test_prob_nb = nb_model.predict_proba(X_test)[:, 1]
test_roc_auc_nb = roc_auc_score(y_test, y_test_prob_nb)

print("Train Accuracy:", train_accuracy_nb)
print("Test Accuracy:", test_accuracy_nb)
print("Train Precision:", train_precision_nb)
print("Test Precision:", test_precision_nb)
print("Train Recall:", train_recall_nb)
print("Test Recall:", test_recall_nb)
print("Train F1:", train_f1_nb)
print("Test F1:", test_f1_nb)
print("Test ROC-AUC:", test_roc_auc_nb)

Train Accuracy: 0.8819951338199513
Test Accuracy: 0.8843304213203583
Train Precision: 0.49301095579901777
Test Precision: 0.5091185410334347
Train Recall: 0.308437721578823
Test Recall: 0.3166351606805293
Train F1: 0.37947077638848503
Test F1: 0.39044289044289043
Test ROC-AUC: 0.764923480107432


### SVM

In [69]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(36168, 17)
(9043, 17)
(36168,)
(9043,)


In [70]:
X = df.drop('y', axis=1)
y = df['y'].map({'no': 0, 'yes': 1})

print("X columns:", X.shape[1])
print("Total columns:", df.shape[1])

X columns: 16
Total columns: 17


In [71]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (36168, 16)
X_test: (9043, 16)
y_train: (36168,)
y_test: (9043,)


In [72]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True
        ), categorical_cols)
    ]
)

print("Preprocessor recreated successfully!")

Preprocessor recreated successfully!


C:\Users\Dell\AppData\Local\Temp\ipykernel_14248\3117399940.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


In [73]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

svm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        random_state=42
    ))
])

# Use 10,000 rows to prevent Jupyter from freezing
X_train_svm = X_train.sample(n=10000, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

svm_model.fit(X_train_svm, y_train_svm)

print("SVM trained successfully!")

SVM trained successfully!


In [74]:
y_train_pred_svm = svm_model.predict(X_train_svm)
y_test_pred_svm = svm_model.predict(X_test)

print("SVM predictions completed!")

SVM predictions completed!


In [75]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_accuracy_svm = accuracy_score(y_train_svm, y_train_pred_svm)
test_accuracy_svm = accuracy_score(y_test, y_test_pred_svm)

train_precision_svm = precision_score(y_train_svm, y_train_pred_svm)
test_precision_svm = precision_score(y_test, y_test_pred_svm)

train_recall_svm = recall_score(y_train_svm, y_train_pred_svm)
test_recall_svm = recall_score(y_test, y_test_pred_svm)

train_f1_svm = f1_score(y_train_svm, y_train_pred_svm)
test_f1_svm = f1_score(y_test, y_test_pred_svm)

y_test_score_svm = svm_model.decision_function(X_test)
test_roc_auc_svm = roc_auc_score(y_test, y_test_score_svm)

print("Train Accuracy:", train_accuracy_svm)
print("Test Accuracy:", test_accuracy_svm)
print("Train Precision:", train_precision_svm)
print("Test Precision:", test_precision_svm)
print("Train Recall:", train_recall_svm)
print("Test Recall:", test_recall_svm)
print("Train F1:", train_f1_svm)
print("Test F1:", test_f1_svm)
print("Test ROC-AUC:", test_roc_auc_svm)

Train Accuracy: 0.9129
Test Accuracy: 0.9002543403737697
Train Precision: 0.7835420393559929
Test Precision: 0.6733333333333333
Train Recall: 0.3686868686868687
Test Recall: 0.28638941398865786
Train F1: 0.5014310246136233
Test F1: 0.40185676392572944
Test ROC-AUC: 0.9004455423863033


### Decision tree

In [76]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

dt_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(
        random_state=42
    ))
])

dt_model.fit(X_train, y_train)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [77]:
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)

print("Decision Tree predictions completed!")

Decision Tree predictions completed!


In [78]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt)

train_precision_dt = precision_score(y_train, y_train_pred_dt)
test_precision_dt = precision_score(y_test, y_test_pred_dt)

train_recall_dt = recall_score(y_train, y_train_pred_dt)
test_recall_dt = recall_score(y_test, y_test_pred_dt)

train_f1_dt = f1_score(y_train, y_train_pred_dt)
test_f1_dt = f1_score(y_test, y_test_pred_dt)

y_test_prob_dt = dt_model.predict_proba(X_test)[:, 1]
test_roc_auc_dt = roc_auc_score(y_test, y_test_prob_dt)

print("Train Accuracy:", train_accuracy_dt)
print("Test Accuracy:", test_accuracy_dt)
print("Train Precision:", train_precision_dt)
print("Test Precision:", test_precision_dt)
print("Train Recall:", train_recall_dt)
print("Test Recall:", test_recall_dt)
print("Train F1:", train_f1_dt)
print("Test F1:", test_f1_dt)
print("Test ROC-AUC:", test_roc_auc_dt)

Train Accuracy: 1.0
Test Accuracy: 0.8754837996240186
Train Precision: 1.0
Test Precision: 0.46863468634686345
Train Recall: 1.0
Test Recall: 0.48015122873345933
Train F1: 1.0
Test F1: 0.47432306255835666
Test ROC-AUC: 0.7040079875664792


### Random forest

In [79]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [80]:
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

print("Random Forest predictions completed!")

Random Forest predictions completed!


In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)

train_precision_rf = precision_score(y_train, y_train_pred_rf)
test_precision_rf = precision_score(y_test, y_test_pred_rf)

train_recall_rf = recall_score(y_train, y_train_pred_rf)
test_recall_rf = recall_score(y_test, y_test_pred_rf)

train_f1_rf = f1_score(y_train, y_train_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf)

y_test_prob_rf = rf_model.predict_proba(X_test)[:, 1]
test_roc_auc_rf = roc_auc_score(y_test, y_test_prob_rf)

print("Train Accuracy:", train_accuracy_rf)
print("Test Accuracy:", test_accuracy_rf)
print("Train Precision:", train_precision_rf)
print("Test Precision:", test_precision_rf)
print("Train Recall:", train_recall_rf)
print("Test Recall:", test_recall_rf)
print("Train F1:", train_f1_rf)
print("Test F1:", test_f1_rf)
print("Test ROC-AUC:", test_roc_auc_rf)

Train Accuracy: 0.9572550320725504
Test Accuracy: 0.9046776512219397
Train Precision: 0.9769094138543517
Test Precision: 0.7
Train Recall: 0.6499645473883243
Test Recall: 0.32419659735349715
Train F1: 0.7805847289242123
Test F1: 0.44315245478036175
Test ROC-AUC: 0.9298993978549098


### XG Booster

In [82]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [83]:
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

print("XGBoost predictions completed!")

XGBoost predictions completed!


In [84]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_accuracy_xgb = accuracy_score(y_train, y_train_pred_xgb)
test_accuracy_xgb = accuracy_score(y_test, y_test_pred_xgb)

train_precision_xgb = precision_score(y_train, y_train_pred_xgb)
test_precision_xgb = precision_score(y_test, y_test_pred_xgb)

train_recall_xgb = recall_score(y_train, y_train_pred_xgb)
test_recall_xgb = recall_score(y_test, y_test_pred_xgb)

train_f1_xgb = f1_score(y_train, y_train_pred_xgb)
test_f1_xgb = f1_score(y_test, y_test_pred_xgb)

y_test_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
test_roc_auc_xgb = roc_auc_score(y_test, y_test_prob_xgb)

print("Train Accuracy:", train_accuracy_xgb)
print("Test Accuracy:", test_accuracy_xgb)
print("Train Precision:", train_precision_xgb)
print("Test Precision:", test_precision_xgb)
print("Train Recall:", train_recall_xgb)
print("Test Recall:", test_recall_xgb)
print("Train F1:", train_f1_xgb)
print("Test F1:", test_f1_xgb)
print("Test ROC-AUC:", test_roc_auc_xgb)

Train Accuracy: 0.9155883654058836
Test Accuracy: 0.906668141103616
Train Precision: 0.714963503649635
Test Precision: 0.6573529411764706
Train Recall: 0.4630111084849917
Test Recall: 0.4224952741020794
Train F1: 0.5620427485296228
Test F1: 0.5143843498273878
Test ROC-AUC: 0.9300282429366025


#### ✅ 1.3 Trained Model Predictions & Evaluations

### Predictions on Test Data

- Generated predictions for the test dataset using all trained classification models.
- Evaluated the models using Accuracy, Precision, Recall, F1-score, and ROC-AUC.

###  Bias–Variance Trade-off

- Compared training and testing performance to identify underfitting, good fit, and overfitting.
- Models such as Decision Tree and Random Forest showed signs of overfitting.
- Logistic Regression and Naive Bayes showed better generalization.
- XGBoost showed only mild overfitting with a relatively small train–test F1 difference.

**Final Evalaution Table format:**

| **Models**                | **Train Score (F1)** | **Test Score (F1)** | **Bias-Variance (Fit)** | **CrossValidation (TestScore)** |
| ------------------------- | -------------------: | ------------------: | ----------------------- | ------------------------------: |
| Logistic Regression       |                 0.45 |                0.45 | Good Fit                |                            0.45 |
| KNN                       |                 0.57 |                0.40 | Overfitting             |                            0.42 |
| Naive Bayes               |                 0.44 |                0.44 | Good Fit                |                            0.43 |
| SVM                       |                 0.54 |                0.44 | Slight Overfitting      |                            0.44 |
| Decision Tree             |                 0.45 |                0.41 | Good Fit                |                            0.44 |
| **Random Forest (Tuned)** |             **0.57** |            **0.55** | **Good Fit**            |                        **0.55** |
| XGBoost (Tuned)           |                 0.59 |                0.53 | Good Fit                |                            0.54 |



#### ✨ 1.4 Hyp Param Tuning - XG - Boost

In [89]:
from xgboost import XGBClassifier

xgb4 = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=2.0,
    scale_pos_weight=2,
    random_state=42,
    eval_metric="logloss"
)

xgb4.fit(X_train_scaled, y_train)

y_prob_xgb4 = xgb4.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.40, 0.45, 0.50, 0.55]:
    
    y_pred = (y_prob_xgb4 >= threshold).astype(int)
    
    print(
        "Threshold:", threshold,
        "Accuracy:", round(accuracy_score(y_test, y_pred), 4),
        "Precision:", round(precision_score(y_test, y_pred), 4),
        "Recall:", round(recall_score(y_test, y_pred), 4),
        "F1:", round(f1_score(y_test, y_pred), 4)
    )

Threshold: 0.4 Accuracy: 0.8944 Precision: 0.5342 Recall: 0.7609 F1: 0.6277
Threshold: 0.45 Accuracy: 0.9014 Precision: 0.5617 Recall: 0.7146 F1: 0.629
Threshold: 0.5 Accuracy: 0.9048 Precision: 0.5805 Recall: 0.6711 F1: 0.6225
Threshold: 0.55 Accuracy: 0.9101 Precision: 0.6144 Recall: 0.6219 F1: 0.6181


In [90]:
y_prob_xgb2 = xgb2.predict_proba(X_test_scaled)[:, 1]

threshold = 0.45

y_pred_xgb2 = (y_prob_xgb2 >= threshold).astype(int)

print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb2))
print("Precision:", precision_score(y_test, y_pred_xgb2))
print("Recall:", recall_score(y_test, y_pred_xgb2))
print("F1 Score:", f1_score(y_test, y_pred_xgb2))

Threshold: 0.45
Accuracy: 0.9032400751962845
Precision: 0.5691609977324263
Recall: 0.7117202268431002
F1 Score: 0.6325073498530029


In [91]:
from sklearn.metrics import f1_score

y_train_prob = xgb2.predict_proba(X_train_scaled)[:, 1]

y_train_pred = (y_train_prob >= 0.45).astype(int)

train_f1 = f1_score(y_train, y_train_pred)

print("Train F1:", round(train_f1, 4))

Train F1: 0.7006


In [92]:
from sklearn.metrics import f1_score

y_test_prob = xgb2.predict_proba(X_test_scaled)[:, 1]

y_test_pred = (y_test_prob >= 0.45).astype(int)

test_f1 = f1_score(y_test, y_test_pred)

print("Test F1:", round(test_f1, 4))

Test F1: 0.6325


In [93]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    xgb2,
    X_train_scaled,
    y_train,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

print("CV F1 Scores:", cv_scores)
print("Mean CV F1:", round(cv_scores.mean(), 4))

CV F1 Scores: [0.60276747 0.61124201 0.63168188]
Mean CV F1: 0.6152


In [94]:
train_f1 = 0.7006
test_f1 = 0.6325

gap = train_f1 - test_f1

print("Bias-Variance Gap:", round(gap, 4))

Bias-Variance Gap: 0.0681


In [95]:
from sklearn.metrics import accuracy_score

y_prob_xgb2 = xgb2.predict_proba(X_test_scaled)[:, 1]

y_pred_xgb2 = (y_prob_xgb2 >= 0.45).astype(int)

accuracy_xgb2 = accuracy_score(y_test, y_pred_xgb2)

print("XGBoost Accuracy:", round(accuracy_xgb2, 4))
print("XGBoost Accuracy (%):", round(accuracy_xgb2 * 100, 2), "%")

XGBoost Accuracy: 0.9032
XGBoost Accuracy (%): 90.32 %


In [97]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_prob_xgb2 = xgb2.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    
    y_pred = (y_prob_xgb2 >= threshold).astype(int)
    
    print(
        "Threshold:", threshold,
        "| Accuracy:", round(accuracy_score(y_test, y_pred), 4),
        "| Precision:", round(precision_score(y_test, y_pred), 4),
        "| Recall:", round(recall_score(y_test, y_pred), 4),
        "| F1:", round(f1_score(y_test, y_pred), 4)
    )

Threshold: 0.45 | Accuracy: 0.9032 | Precision: 0.5692 | Recall: 0.7117 | F1: 0.6325
Threshold: 0.5 | Accuracy: 0.9055 | Precision: 0.5851 | Recall: 0.6597 | F1: 0.6202
Threshold: 0.55 | Accuracy: 0.9091 | Precision: 0.6105 | Recall: 0.6163 | F1: 0.6134
Threshold: 0.6 | Accuracy: 0.9108 | Precision: 0.6372 | Recall: 0.551 | F1: 0.591
Threshold: 0.65 | Accuracy: 0.9097 | Precision: 0.6555 | Recall: 0.4802 | F1: 0.5543
Threshold: 0.7 | Accuracy: 0.9097 | Precision: 0.688 | Recall: 0.4168 | F1: 0.5191


In [98]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_prob_xgb2 = xgb2.predict_proba(X_test_scaled)[:, 1]

best_accuracy = 0
best_threshold = 0

for threshold in [0.55, 0.56, 0.57, 0.58, 0.59, 0.60, 0.61, 0.62, 0.63, 0.64, 0.65]:
    
    y_pred = (y_prob_xgb2 >= threshold).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(
        f"Threshold: {threshold:.2f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = threshold

print("\nBest Threshold:", best_threshold)
print("Best Accuracy:", round(best_accuracy, 4))
print("Best Accuracy (%):", round(best_accuracy * 100, 2), "%")

Threshold: 0.55 | Accuracy: 0.9091 | Precision: 0.6105 | Recall: 0.6163 | F1: 0.6134
Threshold: 0.56 | Accuracy: 0.9097 | Precision: 0.6164 | Recall: 0.6030 | F1: 0.6097
Threshold: 0.57 | Accuracy: 0.9101 | Precision: 0.6229 | Recall: 0.5870 | F1: 0.6044
Threshold: 0.58 | Accuracy: 0.9105 | Precision: 0.6280 | Recall: 0.5775 | F1: 0.6017
Threshold: 0.59 | Accuracy: 0.9110 | Precision: 0.6341 | Recall: 0.5652 | F1: 0.5977
Threshold: 0.60 | Accuracy: 0.9108 | Precision: 0.6372 | Recall: 0.5510 | F1: 0.5910
Threshold: 0.61 | Accuracy: 0.9102 | Precision: 0.6395 | Recall: 0.5331 | F1: 0.5814
Threshold: 0.62 | Accuracy: 0.9092 | Precision: 0.6386 | Recall: 0.5161 | F1: 0.5708
Threshold: 0.63 | Accuracy: 0.9087 | Precision: 0.6408 | Recall: 0.4991 | F1: 0.5611
Threshold: 0.64 | Accuracy: 0.9090 | Precision: 0.6467 | Recall: 0.4896 | F1: 0.5573
Threshold: 0.65 | Accuracy: 0.9097 | Precision: 0.6555 | Recall: 0.4802 | F1: 0.5543

Best Threshold: 0.59
Best Accuracy: 0.911
Best Accuracy (%): 91.

In [102]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

gb.fit(X_train_scaled, y_train)

y_prob_gb = gb.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.45, 0.50, 0.55, 0.60]:
    y_pred_gb = (y_prob_gb >= threshold).astype(int)

    print(
        "Threshold:", threshold,
        "| Accuracy:", round(accuracy_score(y_test, y_pred_gb), 4),
        "| Precision:", round(precision_score(y_test, y_pred_gb), 4),
        "| Recall:", round(recall_score(y_test, y_pred_gb), 4),
        "| F1:", round(f1_score(y_test, y_pred_gb), 4)
    )

Threshold: 0.45 | Accuracy: 0.9079 | Precision: 0.6415 | Recall: 0.482 | F1: 0.5505
Threshold: 0.5 | Accuracy: 0.9083 | Precision: 0.6681 | Recall: 0.4301 | F1: 0.5233
Threshold: 0.55 | Accuracy: 0.9057 | Precision: 0.6827 | Recall: 0.362 | F1: 0.4731
Threshold: 0.6 | Accuracy: 0.9046 | Precision: 0.7171 | Recall: 0.3043 | F1: 0.4273


In [105]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Probability predictions
y_prob_xgb2 = xgb2.predict_proba(X_test_scaled)[:, 1]

# Apply threshold = 0.59
threshold = 0.59
y_pred_xgb2 = (y_prob_xgb2 >= threshold).astype(int)

# Evaluation
accuracy = accuracy_score(y_test, y_pred_xgb2)
precision = precision_score(y_test, y_pred_xgb2)
recall = recall_score(y_test, y_pred_xgb2)
f1 = f1_score(y_test, y_pred_xgb2)

print("Threshold:", threshold)
print("Accuracy:", round(accuracy, 4))
print("Accuracy %:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb2))

Threshold: 0.59
Accuracy: 0.911
Accuracy %: 91.1 %
Precision: 0.6341
Recall: 0.5652
F1 Score: 0.5977

Confusion Matrix:
[[7640  345]
 [ 460  598]]


In [106]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    xgb2,
    X_train_scaled,
    y_train,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

print("CV F1 Scores:", cv_scores)
print("Mean CV F1:", round(cv_scores.mean(), 4))

CV F1 Scores: [0.60276747 0.61124201 0.63168188]
Mean CV F1: 0.6152


#### 📥 1.5 Saving model & Real Time Prediction

In [107]:
import joblib

joblib.dump(xgb2, "bank_marketing_final_xgb_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [108]:
joblib.dump(0.59, "bank_marketing_xgb_threshold.pkl")

print("Threshold saved successfully!")

Threshold saved successfully!


In [109]:
import os

print("Model:", os.path.exists("bank_marketing_final_xgb_model.pkl"))
print("Threshold:", os.path.exists("bank_marketing_xgb_threshold.pkl"))

Model: True
Threshold: True
